The goal of this penultimate notebook is to choose the best model for data derived in nb7. It deals exclusively with model architecture and fine tuning.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import xgboost as xgb

import time

from helpers import preprocessing, submissions, preprocessing

from helpers.utils import (
    stats_helper,
    plotting_helper,
    unsupervised_helper,
    aggregator_helper,
    sklearn_helper,
)
import lightgbm
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier






from helpers.model_builders import bureau_preparation

from helpers.utils.cleaner_helper import plot_categorical_dist, plot_numeric_dist

from lightgbm import LGBMClassifier
import hdbscan
import importlib

In [2]:
data = joblib.load("data/processed/all_tables.jbl")
data.info()

X_train = data[~data["TARGET"].isna()]
X_train = X_train.drop(columns=["SK_ID_CURR"]).reset_index(drop=True)
y_train = X_train.pop("TARGET")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Columns: 473 entries, SK_ID_CURR to onehot_count_NAME_CONTRACT_STATUS_S_y
dtypes: bool(34), category(14), float64(417), int64(7), object(1)
memory usage: 1.1+ GB


We use an adapter that simplifies cross validating models.

In [3]:
def val_model(model, X_train = X_train, y_train = y_train) -> pd.DataFrame:
    """5CV model"""
    return sklearn_helper.stratified_cv_model(
        model,
        X_train,
        y_train,
        scoring=['average_precision', 'roc_auc', 'f1_macro'])

# Dedicated model libraries
## LightGBM

#### Gradient Boosted Tree

In [4]:
val_model(LGBMClassifier(boosting_type='gbdt', verbose=-1))

,average_precision,roc_auc,f1_macro
mean,0.2691,0.7784,0.5143
std,0.0055,0.0043,0.0007


#### Random_forest

In [5]:
val_model(LGBMClassifier(boosting_type='rf', verbose=-1,
        bagging_fraction=0.2, bagging_freq=5))

,average_precision,roc_auc,f1_macro
mean,0.2113,0.7217,0.5545
std,0.0051,0.0043,0.0023


The parameters used, make the RF significantly underperform when compared to gradient boosted trees.
* F1 macro score is improved only due to class imbalance.

#### DART

In [6]:
val_model(LGBMClassifier(boosting_type='dart', verbose=-1))

,average_precision,roc_auc,f1_macro
mean,0.2568,0.7659,0.4857
std,0.0066,0.0038,0.0010


Performance slightly worse in all regards, when compared to gradient boosted trees.

## XGBoost
XGboost provides alternative gradient boosted tree application. More importantly, it has been shown to outperform LightGBM for large datasets.

In [7]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    enable_categorical=True,
)

val_model(model)

,average_precision,roc_auc,f1_macro
mean,0.2496,0.7660,0.5357
std,0.0051,0.0025,0.0038


In [8]:
model = xgb.XGBClassifier(
    objective="reg:logistic",
    enable_categorical=True,
)

val_model(model)

,average_precision,roc_auc,f1_macro
mean,0.2496,0.7660,0.5357
std,0.0051,0.0025,0.0038


# Nan-sensitive models
Models unable to impute nan values will need to rely on an imputer. 

## NAN classification
As data imputation will be introducing noise and bias, it is important to first quantify how much data is missing and whether features need to be discarded.

Models of different architecture are tested with the goal of building an ensemble. 

In [4]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [col for col in num_cols if X_train[col].notna().sum() > 0]


cat_cols = X_train.select_dtypes(include=["category"]).columns.tolist()
cat_cols = [col for col in cat_cols if X_train[col].notna().sum() > 0]


from helpers.utils import sklearn_helper


cat_as_obj_transformer = make_column_transformer(
    (FunctionTransformer(lambda x: x.astype("str")), cat_cols),
    remainder="passthrough",
    verbose_feature_names_out=False,
)
cat_as_obj_transformer.set_output(transform="pandas")

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("str_converter", sklearn_helper.category_transformer(cat_cols, "str")),
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ],
    remainder="drop",
)
display(preprocessor)
X_imputed = preprocessor.fit_transform(X_train)
X_imputed.shape

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['CNT_CHILDREN', 'AMT_INCOME_TOTAL',
                                  'AMT_CREDIT', 'AMT_ANNUITY',
                                  'AMT_GOODS_PRICE',
                                  'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH',
                                  'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
                                  'DAYS_ID_PUBLISH', 'OWN_CAR_AGE',
                                  'CNT_FAM_MEMBERS', 'REGION_RATI...
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['NAME_CONTRACT_TYPE', 'CODE_GENDER',
                                  'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
                                  'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
                                  'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
                                  'WEEKDAY_APPR_PROCESS_START',
                                  'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE',
                                  'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE',
                                  'EMERGENCYSTATE_MODE'])])

(307511, 556)

This preprocessor is then tested on 4 unique structure sklearn models capable of outputting ROC_AUC curves.

In [5]:
sk_models = [
    LogisticRegression(max_iter=1000, random_state=3),
    RandomForestClassifier(random_state=3),
    KNeighborsClassifier(),
    HistGradientBoostingClassifier(random_state=3),
]

def create_pipes(model, preprocessor):
    pipe = Pipeline([("preprocessor", preprocessor), (type(model).__name__, model)])
    return pipe

In [ ]:
for model in sk_models:
    start_time = time.time()

    pipe = create_pipes(model, preprocessor)

    print(f"Validating: {model}")
    display(val_model(pipe, X_train))
    end_time = time.time()


    execution_time = end_time - start_time
    print(f"\nDuration: {execution_time:.2f} seconds")



Validating: LogisticRegression(max_iter=1000, random_state=3)


,average_precision,roc_auc,f1_macro
mean,0.2415,0.7630,0.5036
std,0.0058,0.0048,0.0019



Duration: 238.05 seconds
Validating: RandomForestClassifier(random_state=3)


,average_precision,roc_auc,f1_macro
mean,0.2034,0.7161,0.4803
std,0.0054,0.0035,0.0006



Duration: 2025.50 seconds
Validating: KNeighborsClassifier()


,average_precision,roc_auc,f1_macro
mean,0.1052,0.5802,0.5032
std,0.0015,0.0020,0.0019



Duration: 1859.81 seconds
Validating: HistGradientBoostingClassifier(random_state=3)


,average_precision,roc_auc,f1_macro
mean,0.2656,0.7757,0.5105
std,0.0054,0.0036,0.0017



Duration: 394.11 seconds


* The performance of HistGradientBoostingClassifier is near identical to that of LGBM GBDT. Given that both algorithms are rather similar of implementations of decisions trees, they can be treated as near interchangeable. lgbm will be used due to performance gains on large data.
* LogisticRegression provides comparable performance and trains at acceptable rate.
* KNN performs very poorly due to imbalance in the dataset.
* Random Forest performs comparably to LGBM implementation, but at a much slower training time.

In conclusion, the data imputation and preprocessing techniques have minimal effect on models performance and can be considered appropriate.

#### Smote and KNN
To account for data imbalance and better evaluate kNN performance, SMOTE will be used to generate extra samples.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=5)

from sklearn.

X_smote, y_smote = smote.fit_resample(X_imputed, y_train)

print("resampled X shape:", X_smote.shape)
print("resampled y shape:", y_smote.shape)
val_model(KNeighborsClassifier(), X_smote, y_smote)

resampled X shape: (565372, 556)
resampled y shape: (565372,)


KeyboardInterrupt: 

## Voting Classifier
The best performing models are then investigated along their ROC and AUC curves.

In [ ]:


class_sk_models = [LogisticRegression(max_iter=1000, random_state=3)]
voter = VotingClassifier(
    estimators=[("lgbm_gbdt", LGBMClassifier(boosting_type="gbdt", verbose=-1)),]
    + [
        (type(model).__name__, create_pipes(model, preprocessor))
        for model in class_sk_models
    ],
    voting="soft",
    n_jobs=-1,
)


display(voter)
val_model(voter)

,estimators,"[('lgbm_gbdt', ...), ('LogisticRegression', ...)]"
,voting,'soft'
,weights,None
,n_jobs,-1
,flatten_transform,True
,verbose,False
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100


In [ ]:
val_model(voter)

,average_precision,roc_auc,f1_macro
mean,0.2677,0.7779,0.5028
std,0.0063,0.0045,0.0015


# Final model

In [ ]:



cv_helpers.model_curves(voter, X_train, y_train, cv=5, figsize=(12, 8))